# Local Variation + GNN with Coarsening aware Loss

## Import

In [ ]:
import os
import sys
import random
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# pygsp
from pygsp import graphs

# scipy
import scipy as sp      

# torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# torch geometric
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

# sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

# datasets
data_path = os.path.abspath(os.path.join("..", "data"))
if data_path not in sys.path:
    sys.path.append(data_path)
from cora.load_cora import load_cora_dataset

# plot
import matplotlib.pyplot as plt

# utils
from utils.split_dataset import *
from utils.visualization import *
from utils.coarsening import apply_Loukas_coarsening, create_pygsp_graph

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(1)
np.random.seed(1)


## Global data

- `NODE_IDS`: list of nodes ids
- `FEATURES`: (N x D) matrix of features for each node
- `LABELS`: (N) list of true labels for each node
- `EDGES_IDX`: (M, 2) matrix of edges
- `IDX_MAP`: dict mapping original node IDs to index

In [ ]:
# load dataset and gloa
NODE_IDS, FEATURES, LABELS, EDGES_IDX, IDX_MAP = load_cora_dataset(log_info=False)
LABELS_ENCODED = LabelEncoder().fit_transform(LABELS)

## General functions

### Get coarsened data from Coarsening matrix C

In [ ]:
def get_coarsened_edges_and_features(C: list, Gc: graphs.Graph, features):
    """
    Coarsen edges and features based on the coarsening matrix C;
    the feature matrix (N x D) is made by summing up features from nodes that are coarsened into the same supernode.

    Output:
      - Gc: coarsened edges index (num_edges, 2)
      - features_coarsened: coarsened features (num_nodes_coarsened, num_features)
    """

    idx_map = {} # map fine node -> coarse node 
    for supernode, node in zip(*C.nonzero()):
        idx_map[node] = supernode
        
    # features coarsened
    features_coarsened = C @ features
    
    # edges coarsened
    edges_idx_coarsened = np.array(Gc.get_edge_list()[:2]).T

    return edges_idx_coarsened, features_coarsened


def get_coarsened_edges_features_labels(C: list, Gc: graphs.Graph, features, labels, priority_label=0):
    """
    Coarsen edges and features based on the coarsening matrix C;
    the feature matrix (N x D) is made by summing up features from nodes that are coarsened into the same supernode.

    Output:
      - edges_idx_coarsened: coarsened edges index (num_edges, 2)
      - features_coarsened: coarsened features (num_nodes_coarsened, num_features)
      - labels_coarsened: coarsened labels (num_nodes_coarsened,)
    """

    idx_map = {} # map fine node -> coarse node 
    for supernode, node in zip(*C.nonzero()):
        idx_map[node] = supernode
        
    # features coarsened
    features_coarsened = C @ features

    # edges coarsened
    edges_idx_coarsened = np.array(Gc.get_edge_list()[:2]).T

    # labels coarsened
    num_classes = len(np.unique(labels))
    all_labels = np.zeros((C.shape[0], num_classes), dtype=np.float32)
    for supernode, node in zip(*C.nonzero()):
        all_labels[supernode, labels[node]] += 1

    labels_coarsened = np.zeros(C.shape[0], dtype=np.int64)
    for supernode in range(C.shape[0]):
        mx = np.argmax(all_labels[supernode])
        priority_count = all_labels[supernode, priority_label]
        if priority_count >= mx:
            labels_coarsened[supernode] = priority_label
        else: 
            labels_coarsened[supernode] = mx

    return edges_idx_coarsened, features_coarsened, labels_coarsened

## GNN

### model and training function

In [ ]:
class GCN(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout=.5):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(nfeat, nhid)
        self.conv2 = GCNConv(nhid, nclass)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)
    
    def get_embeddings(self, x, edge_index):
        x = x.to(next(self.parameters()).device)  # send x to the model's device
        edge_index = edge_index.to(x.device) 
        x = F.relu(self.conv1(x, edge_index))
        return x
    
    def reset_parameters(self):
        self.conv1.reset_parameters()
        self.conv2.reset_parameters()

class CoarseningAwareLoss(nn.Module):
    def __init__(self, coarse_weight: float = 1.0):
        """
        Args:
          coarse_weight: weight for the coarsening loss term.
        """
        super().__init__()
        self.coarse_weight = coarse_weight
        self.class_loss = nn.NLLLoss()


    def forward(self,
                output: torch.Tensor,
                embeddings: torch.Tensor,
                labels: torch.Tensor,
                coarsening_matrix: torch.Tensor,
                train_idx: torch.Tensor):
        """
        output: [N, C] log-probabilities (log_softmax).
        embeddings: [N, D] raw features from model.get_embeddings().
        labels: [N] ground-truth class labels.
        coarsening_matrix: [Nc, N] coarsening matrix.
        train_idx: indices of coarsened nodes used for classification loss.
        """
        device = output.device
        N = embeddings.shape[0]

        # 1. Classification loss
        loss_cls = self.class_loss(output[train_idx], labels[train_idx])

        # 2. Embedding normalization
        embeddings_norm = F.normalize(embeddings, p=2, dim=1)

        with torch.no_grad():
            supernodes = torch.zeros(N, dtype=torch.long, device=device)
            for i, j in zip(*coarsening_matrix.nonzero()):
                supernodes[j] = i


        loss_coarse = torch.tensor(0.0, device=device)
        count = 0

        n_sample = 500
        for _ in range(n_sample):
            i, j = random.sample(range(N), 2)
            emb_i = embeddings_norm[i]
            emb_j = embeddings_norm[j]
            sim = F.cosine_similarity(emb_i.unsqueeze(0), emb_j.unsqueeze(0)).squeeze()

            if supernodes[i] == supernodes[j]:
                loss_coarse += 1 - sim
            else:
                loss_coarse += sim
            count += 1

        loss_coarse = loss_coarse / count if count > 0 else torch.tensor(0.0, device=device)
        return loss_cls + self.coarse_weight * loss_coarse

In [ ]:
def create_pyg_data(features, edges_idx, labels) -> Data:
    x = torch.FloatTensor(features.astype(np.float32))
    y = torch.LongTensor(labels)
    edge_index = torch.LongTensor(edges_idx.T)  # expects (2, num_edges)
    edge_index = to_undirected(edge_index)
        
    return Data(x=x, edge_index=edge_index, y=y)

def train_gnn_1_epoch(model: nn.Module, optimizer: optim.Optimizer, criterion: nn.Module, data: Data, embeddings, coarsening_matrix, train_idx: list, val_idx: list):
    """
    Output:
        - train_loss
        - val_loss
        - val_accuracy
    """
    
    data = data.to(next(model.parameters()).device)
    
    model.train()
    optimizer.zero_grad()
    
    output = model(data.x, data.edge_index)
    # embeddings = model.get_embeddings(data.x, data.edge_index)

    # train
    loss = criterion(output, embeddings, data.y, coarsening_matrix, train_idx)
    loss.backward()
    optimizer.step()
    
    # validate
    model.eval()
    with torch.no_grad():
        output = model(data.x, data.edge_index)
        # embeddings = model.get_embeddings(data.x, data.edge_index)

        loss_val = criterion(output, embeddings, data.y, coarsening_matrix, val_idx)
        pred_val = output[val_idx].max(1)[1]
        acc_val = accuracy_score(data.y[val_idx].cpu().numpy(), pred_val.cpu().numpy())

    
    return loss.item(), loss_val.item(), acc_val

def evaluate_model(model: nn.Module, data: Data, test_idx, log_info=True):
    """Evaluate the model on test set."""
    model.eval()
    data = data.to(next(model.parameters()).device)
    
    with torch.no_grad():
        output = model(data.x, data.edge_index)
        pred_test = output[test_idx].max(1)[1]
        acc_test = accuracy_score(data.y[test_idx].cpu().numpy(), pred_test.cpu().numpy())
        
        if log_info:
            print(f'\nTest Accuracy: {acc_test:.4f}')
            print('\nClassification Report:')
            print(classification_report(
                data.y[test_idx].cpu().numpy(), pred_test.cpu().numpy())
            )

    return acc_test, pred_test

## `Coarsen - GNN and Loss`

**workflow:**
```latex
1. Iterate Levels:
    1. generate embeddings of all nodes from GNN(A, X)
    2. coarsen the graph using embeddings as node features
        - small coarsening ~2%
    3. Train GNN on coarsened nodes for E epochs
        - loss is coarsening-aware
    4. fix the new graph with best coarsening from prev iteration
```

In [ ]:
def train_GNN_coarsening_aware_loss(levels: int, epoch_per_level: int, lr=0.01, wd=5e-4, method='variation_neighborhoods', ratio=0.8, similarity_threshold=0.65):
    # model
    nfeat = FEATURES.shape[1]  # 1433
    nhid = 128
    nclass = len(np.unique(LABELS_ENCODED))  # 7
    dropout = 0.1
    model = GCN(nfeat=nfeat, nhid=nhid, nclass=nclass, dropout=dropout).to(device)

    # criterion
    criterion = CoarseningAwareLoss()

    # train data
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)


    features, edges, labels = FEATURES, EDGES_IDX, LABELS_ENCODED
    CC = sp.sparse.csc_matrix(np.eye(FEATURES.shape[0]))

    x, ycrs, yfine, ylosst, ylossv, valacc = [], [], [], [], [], []
    
    for level in range(levels):
        print(f"\tLevel {level + 1:02d}/{levels}")

        data = create_pyg_data(features, edges, labels)
        
        for epoch in range(1):

            # get embeddings from the GNN
            embeddings = model.get_embeddings(data.x, data.edge_index)

            # coarsen the graph using embeddings as features
            G = create_pygsp_graph(edges, data.num_nodes)
            incremental_ratio = np.log(level **(4/3)) / 100 + 0.02
            C, Gc, Call, Gall = apply_Loukas_coarsening(
                G, X=embeddings, method=method, ratio=incremental_ratio, K=50, similarity_threshold=similarity_threshold, max_levels=1, log_info=True
            )

            # train the GNN on the coarsened graph
            edges_c, features_c, labels_c = get_coarsened_edges_features_labels(
                C, Gc, features, labels, priority_label=0
            )
            data_c = create_pyg_data(features_c, edges_c, labels_c)
            train_idx_c, val_idx_c, test_idx_c = create_train_val_test_split(data_c.num_nodes) 
            train_idx_c = torch.LongTensor(train_idx_c).to(device)
            val_idx_c = torch.LongTensor(val_idx_c).to(device)
            test_idx_c = torch.LongTensor(test_idx_c).to(device)

            train_loss, validation_loss, validation_accuracy = train_gnn_1_epoch(
                model, optimizer, criterion, data_c, embeddings, C, train_idx_c, val_idx_c
            )

            print(f"Epoch {epoch + 1:02d}/{epoch_per_level}, Train Loss: {train_loss:.4f}, Validation Loss: {validation_loss:.4f}, Validation Accuracy: {validation_accuracy:.4f}")

            # evaluate the model on the coarsened graph
            acc_test_c, pred_test_c = evaluate_model(model, data_c, test_idx_c, log_info=False)

            # evaluate on original (fine) graph
            C = C @ CC  # shape: ncc x n, type: csc_matrix
            pred_fine = np.array([-1 for _ in range(NODE_IDS.shape[0])], dtype=np.int64)
            tot = 0
            for idx in range(len(test_idx_c)):
                test_supernode = test_idx_c[idx]
                for fine_node in C.getrow(test_supernode).nonzero()[1]:
                    pred_fine[fine_node] = pred_test_c[idx]
                    tot += 1

            sm = sum(pred_fine == LABELS_ENCODED)
            accuracy_fine = sm / tot

            print(f"Test - coarse accuracy: {acc_test_c:.4f}, fine accuracy: {accuracy_fine:.4f}\n")

            # ploting data
            x.append(epoch + 1)
            ycrs.append(acc_test_c)
            yfine.append(accuracy_fine)
            ylosst.append(train_loss)
            ylossv.append(validation_loss)
            valacc.append(validation_accuracy)
        
        features, edges, labels = features_c, edges_c, labels_c
        CC = C

    np.save(f'saves/data_gnn_CoarseningAwareLoss_V2_levels_{levels}_th_{similarity_threshold*100:.0f}_epochs_{epoch_per_level}.npy', {
        'x': x,
        'ycrs': ycrs,
        'yfine': yfine,
        'ylosst': ylosst,
        'ylossv': ylossv,
        'valacc': valacc,
        'description': f'Data obtained using: {ratio=}, {method=}, {similarity_threshold=}, CoarseningAwareLoss() levels approach'
        })
    
    torch.save(model.state_dict(), f'../models/model_gnn_CoarseningAwareLoss_V2_levels_{levels}_th_{similarity_threshold*100:.0f}_epochs_{epoch_per_level}.pt')

# IMPROVEMENT:
# for level in levels:
#     reset_params
#     for epoch in epochs:
#         coarse_1_level() -> Gc
#         train()

## generate data

In [ ]:
from itertools import product

# 25 levels gets to 762 nodes (about 30% of the original graph)
 
levels = [20, 30, 40, 50, 60]
epochs_per_lev = [1, 2, 4, 8, 16]
method = 'variation_neighborhoods'
thresholds = [0.50, 0.60, 0.70, 0.85]


for level, threshold, ep_per_lev in product(levels, thresholds, epochs_per_lev):
    train_GNN_coarsening_aware_loss(levels=level, epoch_per_level=ep_per_lev, method=method, similarity_threshold=threshold)

## plot data

In [ ]:
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray']
levels = [20, 30, 40, 50]

def plot_accuracy_coarse_fine(ycrs, yfine):
    plt.figure(figsize=(10, 5))
    plt.plot(ycrs, label='Coarse Accuracy', color='tab:blue')
    plt.plot(yfine, label='Fine Accuracy', linestyle=':', color='tab:blue')
    plt.legend()
    plt.xlabel('Iteration')
    plt.ylabel('Accuracy')
    plt.title('Coarse vs Fine Accuracy over Iterations')
    plt.tight_layout()
    plt.show()
    # plt.savefig('saves/iterative_custom_loss_accuracy.pdf')

# plot graphs one each independently
# for method, ratio, threshold in product(methods, ratios, thresholds):
    # name = f"data_gnn_CoarseningAwareLoss_threshold_{threshold*100:.0f}_{method}_ratio_{ratio*100:.0f}.npy"
    # data = np.load(f"saves/{name}", allow_pickle=True).item()
    # print(data['description'])
    # plot_training_curves(data['ylosst'], data['ylossv'], data['valacc'])
    # plot_accuracy_coarse_fine(data['ycrs'], data['yfine'])
    
# same ratio and method in the same graph
for level, ep_per_lev in product(levels, epochs_per_lev):
    print(f"Level: {level}, Epochs per Level: {ep_per_lev}")
    plt.figure(figsize=(12, 5))
    for idx, threshold in enumerate(thresholds):
        name = f"data_gnn_CoarseningAwareLoss_V2_levels_{level}_th_{threshold*100:.0f}_epochs_{ep_per_lev}.npy"
        data = np.load(f"saves/{name}", allow_pickle=True).item()
        width = 2 if idx == len(thresholds) - 1 else 1
        plt.plot(data['ycrs'], color=colors[idx], linewidth=width, label=f'Coarse {threshold*100:.0f}%')
        plt.plot(data['yfine'], linestyle=':', color=colors[idx], linewidth=width, label=f'Fine {threshold*100:.0f}%')
    plt.xlabel('Iteration')
    plt.ylabel('Accuracy')
    plt.title('Coarse vs Fine Accuracy over Iterations')
    plt.tight_layout()
    plt.legend()
    plt.show()

#  last accuracy (coarse - fine)

In [ ]:
# same ratio and method in the same graph
for level, threshold in product(levels, thresholds):
    print(f"Level: {level}, Threshold: {threshold}")
    for ep_per_lev in  epochs_per_lev:
        name = f"data_gnn_CoarseningAwareLoss_V2_levels_{level}_th_{threshold*100:.0f}_epochs_{ep_per_lev}.npy"
        data = np.load(f"saves/{name}", allow_pickle=True).item()
        print(f"threshold: {threshold}, Last coarse accuracy: {data['ycrs'][-1]:.4f}, fine accuracy: {data['yfine'][-1]:.4f}")